<a href="https://colab.research.google.com/github/codewithhstyle/Image_based_garbage_classification/blob/main/Image_Based_Garbage_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Data Download, Filtering, and Visualization

In [ ]:

!pip install -q split-folders gradio

import os
import shutil
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import tensorflow as tf

print(" Downloading TrashNet dataset...")

!wget -q https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip

print("Unzipping dataset...")

!unzip -q dataset-resized.zip -d raw_data


source_dir = 'raw_data/dataset-resized'


target_classes = ['glass', 'plastic', 'paper', 'metal']
clean_data_dir = 'filtered_dataset'

os.makedirs(clean_data_dir, exist_ok=True)


for category in target_classes:
    shutil.copytree(os.path.join(source_dir, category),
                    os.path.join(clean_data_dir, category))

print(f"Dataset ready! Filtered classes: {os.listdir(clean_data_dir)}")


plt.figure(figsize=(10, 8))
for i, category in enumerate(target_classes):
    folder_path = os.path.join(clean_data_dir, category)
    random_image = random.choice(os.listdir(folder_path))
    img_path = os.path.join(folder_path, random_image)

    img = mpimg.imread(img_path)
    plt.subplot(2, 2, i+1)
    plt.imshow(img)
    plt.title(category.capitalize())
    plt.axis('off')
plt.tight_layout()
plt.show()

#  Data Splitting and Image Preprocessing Generators

In [ ]:
import splitfolders
from tensorflow.keras.preprocessing.image import ImageDataGenerator


splitfolders.ratio(clean_data_dir, output="dataset_split", seed=42, ratio=(0.7, 0.15, 0.15))


IMG_SIZE = (224, 224)
BATCH_SIZE = 32


train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)


val_test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)


train_generator = train_datagen.flow_from_directory(
    'dataset_split/train', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    'dataset_split/val', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    'dataset_split/test', target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

CLASS_NAMES = list(train_generator.class_indices.keys())
print("Classes learned by the generator:", CLASS_NAMES)

#  Model Building, Training, and Fine-Tuning (MobileNetV2)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping


base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False


model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.2),
    Dense(4, activation='softmax')
])


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


checkpoint = ModelCheckpoint("garbage_classifier.keras", monitor="val_accuracy", save_best_only=True, mode="max")
early_stop = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)


print("\n--- Starting Initial Training ---")
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[checkpoint, early_stop]
)


print("\n--- Starting Fine-Tuning ---")
base_model.trainable = True

for layer in base_model.layers[:-20]:
  layer.trainable = False


model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])


history_finetune = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator,
    callbacks=[checkpoint]
)

#  Model Evaluation on Test Data

In [ ]:

model = tf.keras.models.load_model("garbage_classifier.keras")


print("\nEvaluating on Test Set...")
test_loss, test_acc = model.evaluate(test_generator)
print(f"\n---> Final Test Loss: {test_loss:.4f} <---")
print(f"\n---> Final Test Accuracy: {test_acc * 100:.2f}% <---")

#  Gradio Web Interface Deployment

In [ ]:
import gradio as gr
import tensorflow as tf
from PIL import Image

def predict_garbage(img):

    if img is None:
        return "Please upload an image."


    img = img.convert("RGB")


    img = img.resize((224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)
    img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)


    predictions = model.predict(img_array)


    result = {CLASS_NAMES[i].capitalize(): float(predictions[0][i]) for i in range(4)}
    return result


interface = gr.Interface(
    fn=predict_garbage,
    inputs=[
        gr.Image(label="Upload an Image from device", type="pil"),
    ],
    outputs=gr.Label(num_top_classes=4, label="Prediction Confidence"),
    title="<h1 style='font-weight: 900; font-size: 2rem; color: #1E293B;'>♻️ Image-Based Garbage Classification</h1>",
    description="Upload a picture of garbage to classify it into: Glass, Plastic, Paper, or Metal.",
    flagging_mode="never"
)


interface.launch(debug=True, share=True)